# 02 · Busca: recall e limiar de recusa (v2)

1. Indexa `base/trechos.jsonl` (e5-small + FAISS). **v2:** `metro_elev` com um trecho por estação e linha.
2. Mede recall@1 e recall@3 nos casos `texto`, com destaque para os 8 casos de elevador.
3. Histograma do score do top-1: `texto` × `nao_consta` (os casos `rota` vão para o GTFS, não para a busca).
4. **Limiar como pré-filtro:** a segunda barreira é o LLM ("recuse se não estiver nos trechos"). Critério: o MAIOR limiar com no máximo 1 falsa recusa (1/30) nos casos `texto`. O ponto de F1 máximo aparece só como comparação.
5. Validação leave-one-out do critério.
6. Salva `avaliacao/busca.json` e `avaliacao/limiar.png`. A versão anterior (chunking por linha) está em `busca_v1.json` e `limiar_v1.png`.

In [1]:
import json, random, sys
from pathlib import Path

import numpy as np
import torch

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from sonar_ia.busca import Buscador, MODELO, limiar_prefiltro

buscador = Buscador()
casos = [json.loads(l) for l in (RAIZ / 'avaliacao' / 'casos_teste.jsonl').read_text(encoding='utf-8').splitlines()]
texto = [c for c in casos if c['tipo'] == 'texto']
nao = [c for c in casos if c['tipo'] == 'nao_consta']
print('modelo:', MODELO, '| trechos indexados:', len(buscador.trechos))
print('casos texto:', len(texto), '| nao_consta:', len(nao), '| rota (fora da busca):', sum(c['tipo'] == 'rota' for c in casos))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

modelo: intfloat/multilingual-e5-small | trechos indexados: 108
casos texto: 30 | nao_consta: 15 | rota (fora da busca): 18


## 1. Recall@1 e recall@3 (casos `texto`)

Acerto = algum id de `trechos_esperados` aparece no top-k. A "fonte" de um caso é a fonte do primeiro trecho esperado.

In [2]:
import pandas as pd

fonte_de = {t['id']: t['fonte'] for t in buscador.trechos}
linhas = []
for c in texto:
    top = buscador.buscar(c['pergunta'], k=3)
    ids = [t['id'] for t, _ in top]
    esp = set(c['trechos_esperados'])
    linhas.append({
        'id': c['id'], 'fonte': fonte_de[c['trechos_esperados'][0]],
        'hit1': ids[0] in esp, 'hit3': any(i in esp for i in ids),
        'top3': [(i, round(s, 3)) for i, (_, s) in zip(ids, top)], 'esperados': c['trechos_esperados'],
        'pergunta': c['pergunta'],
    })
res = pd.DataFrame(linhas)
recall1, recall3 = res.hit1.mean(), res.hit3.mean()
print(f'recall@1 = {recall1:.3f} ({res.hit1.sum()}/{len(res)})')
print(f'recall@3 = {recall3:.3f} ({res.hit3.sum()}/{len(res)})')

por_fonte = res.groupby('fonte').agg(casos=('id', 'count'), recall1=('hit1', 'mean'), recall3=('hit3', 'mean')).round(3)
print('\nPor fonte:'); print(por_fonte.to_string())

elev = res[res.fonte == 'metro_elev']
print(f'\n>>> DESTAQUE: {len(elev)} casos de elevador (metro_elev)  recall@1 = {elev.hit1.mean():.3f}  recall@3 = {elev.hit3.mean():.3f}')
resto = res[res.fonte != 'metro_elev']
print(f'>>> demais {len(resto)} casos texto          recall@1 = {resto.hit1.mean():.3f}  recall@3 = {resto.hit3.mean():.3f}')

recall@1 = 0.767 (23/30)
recall@3 = 0.967 (29/30)

Por fonte:
                casos  recall1  recall3
fonte                                  
cptm_acess          2     1.00      1.0
dec_caoguia         5     0.40      1.0
lbi                 4     0.75      1.0
lei_caoguia         1     1.00      1.0
metro_elev          8     1.00      1.0
metro_pref          1     1.00      1.0
smped_guia          4     0.75      1.0
sptrans_manual      5     0.60      0.8

>>> DESTAQUE: 8 casos de elevador (metro_elev)  recall@1 = 1.000  recall@3 = 1.000
>>> demais 22 casos texto          recall@1 = 0.682  recall@3 = 0.955


### Casos que erraram (top-3 recuperado, com score)

In [3]:
def mostrar(df):
    for r in df.itertuples():
        print(f"{r.id} [{r.fonte}] hit@1={r.hit1} hit@3={r.hit3}")
        print('   pergunta :', r.pergunta)
        print('   esperados:', r.esperados)
        print('   top-3    :', r.top3)

errou1 = res[~res.hit1]
print(f'ERRARAM NO TOP-1 ({len(errou1)}):\n')
mostrar(errou1)
print(f'\nERRARAM TAMBÉM NO TOP-3 ({(~res.hit3).sum()}):', list(res[~res.hit3].id))

ERRARAM NO TOP-1 (7):

texto_11 [dec_caoguia] hit@1=False hit@3=True
   pergunta : o motorista ou o metrô pode cobrar a mais por causa do meu cão-guia?
   esperados: ['dec_caoguia_001']
   top-3    : [('smped_guia_004', 0.882), ('dec_caoguia_001', 0.872), ('dec_caoguia_006', 0.866)]
texto_12 [dec_caoguia] hit@1=False hit@3=True
   pergunta : quais documentos o cao guia tem que ter pra andar de transporte?
   esperados: ['dec_caoguia_003', 'smped_guia_004']
   top-3    : [('sptrans_manual_015', 0.874), ('smped_guia_003', 0.869), ('smped_guia_004', 0.866)]
texto_14 [dec_caoguia] hit@1=False hit@3=True
   pergunta : onde meu cachorro guia fica sentado dentro do onibus ou do metrô?
   esperados: ['dec_caoguia_001']
   top-3    : [('smped_guia_004', 0.858), ('dec_caoguia_001', 0.849), ('sptrans_manual_015', 0.849)]
texto_15 [sptrans_manual] hit@1=False hit@3=True
   pergunta : se o motorista ver uma pessoa cega no ponto, ele é obrigado a parar?
   esperados: ['sptrans_manual_005']
   top-3 

### Elevador (`metro_elev`): todos os 8 casos, com a margem top1 - top2

In [4]:
margens = []
for r in elev.itertuples():
    m = r.top3[0][1] - r.top3[1][1]
    margens.append(m)
    print(f"{r.id} hit@1={r.hit1} margem top1-top2 = {m:+.3f}   top-3 = {r.top3}   esperados = {r.esperados}")
print(f'\nmargem média top1-top2 nos elevadores = {np.mean(margens):.4f}')

texto_01 hit@1=True margem top1-top2 = +0.020   top-3 = [('metro_elev_l1-azul_tucuruvi', 0.883), ('metro_elev_l1-azul_santa-cruz', 0.863), ('metro_elev_l1-azul_carandiru', 0.863)]   esperados = ['metro_elev_l1-azul_tucuruvi']
texto_02 hit@1=True margem top1-top2 = +0.005   top-3 = [('metro_elev_l1-azul_paraiso', 0.885), ('metro_elev_l1-azul_parada-inglesa', 0.88), ('metro_elev_l1-azul_portuguesa-tiete', 0.879)]   esperados = ['metro_elev_l1-azul_paraiso', 'metro_elev_l2-verde_paraiso']
texto_03 hit@1=True margem top1-top2 = +0.005   top-3 = [('metro_elev_l1-azul_ana-rosa', 0.907), ('metro_elev_l2-verde_ana-rosa', 0.902), ('metro_elev_l3-vermelha_bras', 0.873)]   esperados = ['metro_elev_l1-azul_ana-rosa', 'metro_elev_l2-verde_ana-rosa']
texto_04 hit@1=True margem top1-top2 = +0.015   top-3 = [('metro_elev_l1-azul_armenia', 0.895), ('metro_elev_l3-vermelha_carrao', 0.88), ('metro_elev_l3-vermelha_penha', 0.872)]   esperados = ['metro_elev_l1-azul_armenia']
texto_05 hit@1=True margem top

## 2. Distribuição do score do top-1: `texto` × `nao_consta`

In [5]:
def top1(cs):
    return np.array([buscador.buscar(c['pergunta'], k=1)[0][1] for c in cs])

s_texto, s_nao = top1(texto), top1(nao)
print(f'texto      : n={len(s_texto)}  min={s_texto.min():.3f}  mediana={np.median(s_texto):.3f}  max={s_texto.max():.3f}')
print(f'nao_consta : n={len(s_nao)}  min={s_nao.min():.3f}  mediana={np.median(s_nao):.3f}  max={s_nao.max():.3f}')

texto      : n=30  min=0.852  mediana=0.884  max=0.938
nao_consta : n=15  min=0.822  mediana=0.858  max=0.878


## 3. Limiar como pré-filtro

Regra: recusar se `score_top1 < limiar`. O limiar só filtra o óbvio; o LLM é a segunda barreira. Por isso o critério privilegia **não recusar perguntas respondíveis**: o **maior limiar com no máximo 1 falsa recusa em 30** (`texto` recusado). Falsa recusa = pergunta `texto` recusada; recusa correta = `nao_consta` recusado.

In [6]:
limiar = limiar_prefiltro(s_texto, max_falsas=1)
falsa_recusa = int((s_texto < limiar).sum())
recusa_correta = int((s_nao < limiar).sum())
tp, fp, fn = recusa_correta, falsa_recusa, len(s_nao) - recusa_correta
prec = tp / (tp + fp) if tp + fp else 0.0
rec = tp / (tp + fn)
f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
print(f'limiar (pré-filtro) = {limiar:.4f}')
print(f'falsa recusa   = {falsa_recusa}/{len(s_texto)} ({falsa_recusa/len(s_texto):.1%})')
print(f'recusa correta = {recusa_correta}/{len(s_nao)} ({recusa_correta/len(s_nao):.1%})')
print(f'precisão da recusa = {prec:.3f}  recall da recusa = {rec:.3f}  F1 = {f1:.3f}')

limiar (pré-filtro) = 0.8537
falsa recusa   = 1/30 (3.3%)
recusa correta = 6/15 (40.0%)
precisão da recusa = 0.857  recall da recusa = 0.400  F1 = 0.545


### Tabela limiar × (falsa recusa, recusa correta), 6 pontos entre 0,82 e 0,90, e o ponto de F1 máximo (só comparação)

In [7]:
def contagens(lim):
    return int((s_texto < lim).sum()), int((s_nao < lim).sum())

scores = np.concatenate([s_texto, s_nao])
rotulo = np.array([0] * len(s_texto) + [1] * len(s_nao))

def f1_em(lim):
    prev = scores < lim
    tp_ = int((prev & (rotulo == 1)).sum()); fp_ = int((prev & (rotulo == 0)).sum()); fn_ = int((~prev & (rotulo == 1)).sum())
    p = tp_ / (tp_ + fp_) if tp_ + fp_ else 0.0
    r = tp_ / (tp_ + fn_) if tp_ + fn_ else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

ordenados = np.unique(scores)
candidatos = np.concatenate([[ordenados[0] - 1e-3], (ordenados[:-1] + ordenados[1:]) / 2, [ordenados[-1] + 1e-3]])
f1s = np.array([f1_em(l)[2] for l in candidatos])
empatados = candidatos[np.isclose(f1s, f1s.max())]
lim_f1 = float((empatados.min() + empatados.max()) / 2)
p_f1, r_f1, f1_f1 = f1_em(lim_f1)

linhas = [(f'{l:.3f}', *contagens(l)) for l in np.linspace(0.82, 0.90, 6)]
linhas.append((f'{limiar:.4f}  <- ESCOLHIDO (pré-filtro)', falsa_recusa, recusa_correta))
linhas.append((f'{lim_f1:.4f}  <- F1 máximo (comparação, F1={f1_f1:.3f})', *contagens(lim_f1)))
tab = pd.DataFrame(linhas, columns=['limiar', 'falsa recusa (de 30)', 'recusa correta (de 15)'])
print(tab.to_string(index=False))

                                     limiar  falsa recusa (de 30)  recusa correta (de 15)
                                      0.820                     0                       0
                                      0.836                     0                       2
                                      0.852                     0                       5
                                      0.868                     6                      11
                                      0.884                    15                      15
                                      0.900                    23                      15
          0.8537  <- ESCOLHIDO (pré-filtro)                     1                       6
0.8739  <- F1 máximo (comparação, F1=0.743)                     7                      13


### Validação leave-one-out

Para cada caso (`texto` + `nao_consta`) o limiar é escolhido pelo mesmo critério **sem esse caso** (no máximo 1 falsa recusa nos `texto` restantes) e o caso é classificado com ele.

In [8]:
falsas_loo, corretas_loo, limiares_loo = 0, 0, []
for i, s in enumerate(s_texto):
    lim_i = limiar_prefiltro(np.delete(s_texto, i), max_falsas=1)
    limiares_loo.append(lim_i)
    falsas_loo += int(s < lim_i)
for j, s in enumerate(s_nao):
    lim_j = limiar_prefiltro(s_texto, max_falsas=1)   # o critério só usa os `texto`: tirar um nao_consta não muda o limiar
    limiares_loo.append(lim_j)
    corretas_loo += int(s < lim_j)
print(f'limiar LOO: de {min(limiares_loo):.4f} a {max(limiares_loo):.4f} (completo: {limiar:.4f})')
print(f'falsa recusa estimada (LOO)   = {falsas_loo}/{len(s_texto)} ({falsas_loo/len(s_texto):.1%})')
print(f'recusa correta estimada (LOO) = {corretas_loo}/{len(s_nao)} ({corretas_loo/len(s_nao):.1%})')

limiar LOO: de 0.8537 a 0.8577 (completo: 0.8537)
falsa recusa estimada (LOO)   = 2/30 (6.7%)
recusa correta estimada (LOO) = 6/15 (40.0%)


In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

bins = np.linspace(min(scores.min(), 0.7) - 0.01, scores.max() + 0.01, 25)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.hist(s_texto, bins=bins, alpha=0.75, label=f'texto (n={len(s_texto)})', color='#1f77b4')
ax.hist(s_nao, bins=bins, alpha=0.75, label=f'nao_consta (n={len(s_nao)})', color='#d62728')
ax.axvline(limiar, color='black', linestyle='-', label=f'limiar pré-filtro = {limiar:.3f}  (falsa recusa {falsa_recusa}/30, recusa correta {recusa_correta}/15)')
ax.axvline(lim_f1, color='gray', linestyle='--', label=f'F1 máximo = {lim_f1:.3f}  (só comparação)')
ax.set_xlabel('score do top-1 (cosseno, e5-small)'); ax.set_ylabel('nº de perguntas')
ax.set_title('Score do trecho mais parecido: respondíveis × sem resposta (v2, 1 trecho por estação)')
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'limiar.png', dpi=130)
plt.close(fig)
print('salvo: avaliacao/limiar.png')

salvo: avaliacao/limiar.png


### Perguntas em cada lado do limiar e pegadinhas

In [10]:
print('nao_consta (score do top-1, trecho recuperado):')
for c, s in sorted(zip(nao, s_nao), key=lambda x: -x[1]):
    marca = 'PASSA ' if s >= limiar else 'recusa'
    top = buscador.buscar(c['pergunta'], k=1)[0][0]['id']
    print(f"  {marca} {s:.3f} {c['id']:<14} top1={top:<36} {c['pergunta']}")
print('\ntexto recusados indevidamente (score < limiar):')
for c, s in zip(texto, s_texto):
    if s < limiar:
        print(f"  RECUSA {s:.3f} {c['id']:<10} {c['pergunta']}")
print('\nPegadinhas de elevador (estações de linhas fora do PDF):')
score_nao = {c['id']: float(s) for c, s in zip(nao, s_nao)}
for id_ in ('nao_consta_01', 'nao_consta_02', 'nao_consta_03', 'nao_consta_14'):
    c = next(c for c in nao if c['id'] == id_)
    top = buscador.buscar(c['pergunta'], k=3)
    print(f"{id_}: {c['pergunta']!r}  top-1 = {score_nao[id_]:.3f}  ({'PASSA' if score_nao[id_] >= limiar else 'recusada'})")
    for t, sc in top:
        print(f'    {sc:.3f} {t["id"]}')

nao_consta (score do top-1, trecho recuperado):
  PASSA  0.878 nao_consta_06  top1=metro_elev_l3-vermelha_tatuape       hoje tem elevador quebrado na estação tatuapé?
  PASSA  0.877 nao_consta_12  top1=smped_guia_004                       como faço pra conseguir um cão guia, quanto custa?
  PASSA  0.873 nao_consta_01  top1=metro_elev_l1-azul_luz               a estação faria lima da linha 4 amarela tem elevador?
  PASSA  0.872 nao_consta_05  top1=metro_elev_l1-azul_se                o elevador da estaçao sé ta funcionando agora?
  PASSA  0.865 nao_consta_09  top1=metro_elev_l1-azul_santa-cruz        a estação carioca do metrô do rio de janeiro tem elevador?
  PASSA  0.864 nao_consta_04  top1=cptm_acess_001                       o que tem de acessibilidade na estação interlagos da linha 9?
  PASSA  0.860 nao_consta_02  top1=metro_elev_l1-azul_liberdade         tem elevador na estação adolfo pinheiro da linha 5 lilás?
  PASSA  0.858 nao_consta_03  top1=metro_elev_l3-vermelha_se          

nao_consta_01: 'a estação faria lima da linha 4 amarela tem elevador?'  top-1 = 0.873  (PASSA)
    0.873 metro_elev_l1-azul_luz
    0.866 metro_elev_l3-vermelha_republica
    0.862 metro_elev_l3-vermelha_belem
nao_consta_02: 'tem elevador na estação adolfo pinheiro da linha 5 lilás?'  top-1 = 0.860  (PASSA)
    0.860 metro_elev_l1-azul_liberdade
    0.859 metro_elev_l3-vermelha_marechal-deodoro
    0.858 metro_elev_l1-azul_ana-rosa
nao_consta_03: 'a estaçao carapicuiba da linha 8 diamante é acessivel pra cego?'  top-1 = 0.858  (PASSA)
    0.858 metro_elev_l3-vermelha_se
    0.856 sptrans_manual_012
    0.855 sptrans_manual_014
nao_consta_14: 'tem embarque preferencial na estação tucuruvi?'  top-1 = 0.851  (recusada)
    0.851 metro_elev_l1-azul_tucuruvi
    0.843 metro_pref_001
    0.836 metro_elev_l3-vermelha_patriarca


## 4. Salvar `avaliacao/busca.json`

In [11]:
extras = {
    'margem_media_top1_top2_elevador': round(float(np.mean(margens)), 4),
    'pegadinhas': {k: round(score_nao[k], 4) for k in ('nao_consta_01', 'nao_consta_02', 'nao_consta_03')},
    'falsa_recusa_no_limiar_escolhido': falsa_recusa,
    'recusa_correta_no_limiar_escolhido': recusa_correta,
    'limiar_prefiltro': limiar,
    'falsa_recusa_prefiltro': falsa_recusa,
    'recusa_correta_prefiltro': recusa_correta,
    'recall@1_elevador': round(float(elev.hit1.mean()), 4),
    'loo': {'falsa_recusa': falsas_loo, 'recusa_correta': corretas_loo},
    'f1_maximo_comparacao': {'limiar': round(lim_f1, 4), 'f1': round(float(f1_f1), 4), 'precisao': round(float(p_f1), 4),
                             'recall_recusa': round(float(r_f1), 4), 'falsa_recusa': contagens(lim_f1)[0],
                             'recusa_correta': contagens(lim_f1)[1]},
}
resultado = {
    'recall@1': round(float(recall1), 4), 'recall@3': round(float(recall3), 4),
    'recall_por_fonte': {f: {'casos': int(r.casos), 'recall@1': float(r.recall1), 'recall@3': float(r.recall3)} for f, r in por_fonte.iterrows()},
    'criterio_limiar': 'maior limiar com no máximo 1 falsa recusa em 30 casos texto (pré-filtro)',
    'limiar': limiar, 'f1': round(float(f1), 4), 'precisao': round(float(prec), 4), 'recall_recusa': round(float(rec), 4),
    'modelo_embedding': MODELO,
    'extras': extras,
}
(RAIZ / 'avaliacao' / 'busca.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(resultado, ensure_ascii=False, indent=2))

{
  "recall@1": 0.7667,
  "recall@3": 0.9667,
  "recall_por_fonte": {
    "cptm_acess": {
      "casos": 2,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "dec_caoguia": {
      "casos": 5,
      "recall@1": 0.4,
      "recall@3": 1.0
    },
    "lbi": {
      "casos": 4,
      "recall@1": 0.75,
      "recall@3": 1.0
    },
    "lei_caoguia": {
      "casos": 1,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "metro_elev": {
      "casos": 8,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "metro_pref": {
      "casos": 1,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "smped_guia": {
      "casos": 4,
      "recall@1": 0.75,
      "recall@3": 1.0
    },
    "sptrans_manual": {
      "casos": 5,
      "recall@1": 0.6,
      "recall@3": 0.8
    }
  },
  "criterio_limiar": "maior limiar com no máximo 1 falsa recusa em 30 casos texto (pré-filtro)",
  "limiar": 0.8537,
  "f1": 0.5455,
  "precisao": 0.8571,
  "recall_recusa": 0.4,
  "modelo_embedding": "int

## 5. Comparação v1 (chunking por linha) × v2 (um trecho por estação)

In [12]:
v1 = json.loads((RAIZ / 'avaliacao' / 'busca_v1.json').read_text(encoding='utf-8'))
v2 = resultado
e1, e2 = v1['extras'], v2['extras']
tabela = pd.DataFrame([
    ('recall@1 (30 texto)', v1['recall@1'], v2['recall@1']),
    ('recall@3 (30 texto)', v1['recall@3'], v2['recall@3']),
    ('recall@1 dos 8 elevadores', e1['recall@1_elevador'], e2['recall@1_elevador']),
    ('margem média top1-top2 (elevadores)', e1['margem_media_top1_top2_elevador'], e2['margem_media_top1_top2_elevador']),
    ('score nao_consta_01 (Faria Lima)', e1['pegadinhas']['nao_consta_01'], e2['pegadinhas']['nao_consta_01']),
    ('score nao_consta_02 (Adolfo Pinheiro)', e1['pegadinhas']['nao_consta_02'], e2['pegadinhas']['nao_consta_02']),
    ('score nao_consta_03 (Carapicuíba)', e1['pegadinhas']['nao_consta_03'], e2['pegadinhas']['nao_consta_03']),
    ('limiar pré-filtro (máx. 1 falsa recusa)', e1['limiar_prefiltro'], e2['limiar_prefiltro']),
    ('falsa recusa nesse limiar (de 30)', e1['falsa_recusa_prefiltro'], e2['falsa_recusa_prefiltro']),
    ('recusa correta nesse limiar (de 15)', e1['recusa_correta_prefiltro'], e2['recusa_correta_prefiltro']),
    ('[v1 publicado] limiar de F1 máximo', v1['limiar'], e2['f1_maximo_comparacao']['limiar']),
    ('   falsa recusa / recusa correta nesse limiar', f"{e1['falsa_recusa_no_limiar_escolhido']}/30, {e1['recusa_correta_no_limiar_escolhido']}/15",
     f"{e2['f1_maximo_comparacao']['falsa_recusa']}/30, {e2['f1_maximo_comparacao']['recusa_correta']}/15"),
], columns=['métrica', 'v1', 'v2'])
print(tabela.to_string(index=False))

                                      métrica          v1          v2
                          recall@1 (30 texto)      0.7667      0.7667
                          recall@3 (30 texto)      0.9667      0.9667
                    recall@1 dos 8 elevadores         1.0         1.0
          margem média top1-top2 (elevadores)      0.0108       0.015
             score nao_consta_01 (Faria Lima)      0.8667      0.8733
        score nao_consta_02 (Adolfo Pinheiro)      0.8574      0.8602
            score nao_consta_03 (Carapicuíba)      0.8582      0.8582
      limiar pré-filtro (máx. 1 falsa recusa)      0.8537      0.8537
            falsa recusa nesse limiar (de 30)           1           1
          recusa correta nesse limiar (de 15)           5           6
           [v1 publicado] limiar de F1 máximo      0.8671      0.8739
   falsa recusa / recusa correta nesse limiar 8/30, 14/15 7/30, 13/15


## 6. Tabela v1 × v2 e justificativa do limiar

| métrica | v1 (um trecho por linha) | v2 (um trecho por estação e linha) |
|---|---|---|
| recall@1 (30 texto) | 0,767 | 0,767 |
| recall@3 (30 texto) | 0,967 | 0,967 |
| recall@1 dos 8 elevadores | 1,000 | 1,000 |
| margem média top1-top2 (elevadores) | 0,0108 | 0,0150 |
| score nao_consta_01 (Faria Lima) | 0,8667 | 0,8733 |
| score nao_consta_02 (Adolfo Pinheiro) | 0,8574 | 0,8602 |
| score nao_consta_03 (Carapicuíba) | 0,8582 | 0,8582 |
| limiar pré-filtro (máx. 1 falsa recusa em 30) | 0,8537 | 0,8537 |
| falsa recusa nesse limiar | 1/30 | 1/30 |
| recusa correta nesse limiar | 5/15 | 6/15 |
| (comparação) limiar de F1 máximo | 0,8671 (8/30 falsas, 14/15 corretas) | 0,8739 (7/30 falsas, 13/15 corretas) |

Os dois recalls usam o gabarito corrigido do `texto_08` (a v1 original dava 0,733 e 7/8 em elevador).

**O chunking por estação quase não mudou a busca.** O recall é idêntico e os 8 elevadores já acertavam o top-1 na v1 depois da correção do gabarito. A margem entre o 1º e o 2º colocado sobe de 0,0108 para 0,0150, o que é pouco. Os scores das pegadinhas não caem, **sobem**: Faria Lima passa de 0,8667 para 0,8733. A hipótese de que agrupar ~15 estações por trecho diluía o embedding **não explica** o problema: com um trecho por estação, a pergunta sobre uma estação que não existe na base ainda recupera um trecho de estação com score alto (Faria Lima → `metro_elev_l1-azul_luz`, que contém "Conexão Linha 4-Amarela"; Adolfo Pinheiro → `metro_elev_l1-azul_liberdade`; Carapicuíba → `metro_elev_l3-vermelha_se`). O e5-small valoriza o vocabulário do domínio ("estação", "elevador") mais que o nome próprio da estação, e isso o chunking não corrige.

**Escolha do limiar.** Como o limiar é só um pré-filtro (a segunda barreira é o LLM), o critério é o maior limiar com no máximo 1 falsa recusa em 30: **0,8537**, que é o segundo menor score dos casos `texto` (arredondado para baixo na 4ª casa). Nesse ponto: falsa recusa 1/30 (só o texto_09, 0,852), recusa correta 6/15, precisão da recusa 0,857, recall da recusa 0,400. O ponto de F1 máximo (0,8739, F1 = 0,743) recusaria 7 das 30 respondíveis (23%), o que um pré-filtro não deve fazer. A tabela do notebook mostra a troca: de 0,852 para 0,868 a recusa correta vai de 5 para 11 de 15, mas a falsa recusa vai de 0 para 6 de 30.

**Validação leave-one-out.** Tirando cada caso e recalculando o limiar, a falsa recusa estimada é de **2/30 (6,7%)** e a recusa correta estimada, de **6/15 (40%)**. O limiar varia só de 0,8537 a 0,8577, mas o critério depende dos dois menores scores de `texto`, então é sensível a dois casos. A taxa de falsa recusa real fica em 3% a 7%.

**O que o pré-filtro faz e o que sobra para o LLM.** Recusados pelo limiar (6/15): show (0,822), corrida de Uber (0,832), preço da passagem (0,837), horário do próximo ônibus (0,837), embarque preferencial no Tucuruvi (0,851) e piso tátil (0,853). **Passam (9/15)**: os 3 de estações de linhas fora do PDF (Faria Lima 0,873, Adolfo Pinheiro 0,860, Carapicuíba 0,858), mais Interlagos (0,864), os dois de tempo real (elevador da Sé 0,872 e Tatuapé 0,878), o metrô do Rio (0,865), o custo do cão-guia (0,877) e o telefone do Metrô (0,857). Esses 9 dependem de o LLM recusar quando os trechos não respondem, e é isso que as etapas 4 e 5 vão medir (taxa de alucinação).

**Limite conhecido da busca (para o relatório): texto_19.** "Quando eu for guiado por alguém, em que braço devo segurar?" erra até no top-3: o trecho certo (`sptrans_manual_004`, "acompanhe a pessoa com deficiência visual oferecendo o braço oposto à bengala") não aparece, e os três primeiros são `sptrans_manual_005` (0,854), `sptrans_manual_011` (0,854) e `sptrans_manual_003` (0,850). A pergunta diz "ser guiado" e o trecho diz "acompanhe… oferecendo o braço", vocabulário diferente para o mesmo conceito, o que é um limite do e5-small. Correção proposta: modelo de embedding maior (e5-base ou bge-m3) ou expansão de consulta. Não corrigido aqui.